In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [2]:
DATA_URL = (
    "https://raw.githubusercontent.com/vyuan2037/"
    "ds-intern-challenge/main/sample-data/product_usage_events.csv"
)

df_raw = pd.read_csv(DATA_URL)

print(f"Raw dataset shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

Raw dataset shape: 41 rows × 12 columns


In [3]:
df_raw.head()

,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
0,2026-08-01,Sales,Lead summary,email,42,35,29,3,8.5,0.74,4.1,normal day
1,2026-08-01,Sales,Lead summary,manual,18,12,8,2,6.0,0.61,3.8,normal day
2,2026-08-01,Support,Reply draft,queue,55,48,39,6,4.5,0.82,4.0,normal day
3,2026-08-01,Support,Reply draft,manual,11,8,5,1,3.0,0.68,NaN,missing rating
4,2026-08-01,Product,Feedback clustering,csv upload,12,9,6,2,14.0,0.59,3.6,small sample


#Checking Basic structure

In [4]:
print(f"Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"\nColumns:\n{df_raw.columns.tolist()}")

summary = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_%": (df_raw.isna().mean() * 100).round(1),
    "unique_values": df_raw.nunique(dropna=True)
})

display(summary)

Shape: 41 rows × 12 columns

Columns:
['date', 'team', 'workflow', 'source', 'sessions', 'completed', 'accepted_output', 'flagged_for_review', 'avg_minutes_saved', 'median_confidence', 'user_rating', 'notes']


,dtype,missing,missing_%,unique_values
date,object,0,0.0,7
team,object,0,0.0,4
workflow,object,0,0.0,3
source,object,0,0.0,4
sessions,int64,0,0.0,31
completed,int64,0,0.0,29
accepted_output,int64,0,0.0,26
flagged_for_review,int64,0,0.0,11
avg_minutes_saved,float64,0,0.0,35
median_confidence,float64,1,2.4,31


#Missing-value audit

In [5]:
missing_audit = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(1),
    "data_type": df_raw.dtypes.astype(str)
}).sort_values("missing_count", ascending=False)

display(missing_audit)

,missing_count,missing_pct,data_type
median_confidence,1,2.4,float64
user_rating,1,2.4,float64
date,0,0.0,object
team,0,0.0,object
source,0,0.0,object
workflow,0,0.0,object
sessions,0,0.0,int64
completed,0,0.0,int64
flagged_for_review,0,0.0,int64
accepted_output,0,0.0,int64


In [6]:
rows_with_missing = df_raw[df_raw.isna().any(axis=1)].copy()

print(f"Rows containing at least one missing value: {len(rows_with_missing)}")
display(rows_with_missing)

Rows containing at least one missing value: 2


,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
3,2026-08-01,Support,Reply draft,manual,11,8,5,1,3.0,0.68,NaN,missing rating
30,2026-08-05,Product,Feedback clustering,manual,9,7,4,1,11.0,NaN,3.5,confidence missing as text


#Duplicates Check

In [7]:
exact_duplicates = df_raw[df_raw.duplicated(keep=False)].sort_values(
    by=df_raw.columns.tolist()
)

print(f"Exact duplicate rows, including repeated copies: {len(exact_duplicates)}")
display(exact_duplicates)

Exact duplicate rows, including repeated copies: 0


,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes


In [8]:
duplicate_key = [col for col in df_raw.columns if col != "notes"]

potential_duplicates = (
    df_raw[df_raw.duplicated(subset=duplicate_key, keep=False)]
    .sort_values(["date", "team", "workflow", "source"])
)

print(f"Potential duplicate-export rows: {len(potential_duplicates)}")
display(potential_duplicates)

Potential duplicate-export rows: 2


,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
24,2026-08-05,Sales,Lead summary,email,140,126,119,2,12.0,0.95,4.9,traffic spike from demo account
25,2026-08-05,Sales,Lead summary,email,140,126,119,2,12.0,0.95,4.9,duplicate export row


#Inconsistent labels and notes


In [9]:
categorical_cols = ["team", "workflow", "source", "notes"]

for col in categorical_cols:
    print(f"\n### {col.upper()} | {df_raw[col].nunique(dropna=False)} unique values")
    display(df_raw[col].value_counts(dropna=False).rename("count").to_frame())


### TEAM | 4 unique values


,count
team,
Sales,14
Support,13
Product,13
product,1



### WORKFLOW | 3 unique values


,count
workflow,
Lead summary,14
Feedback clustering,14
Reply draft,13



### SOURCE | 4 unique values


,count
source,
manual,19
email,8
queue,7
csv upload,7



### NOTES | 9 unique values


,count
notes,
normal day,27
new prompt version started,6
small sample,2
missing rating,1
team casing differs,1
traffic spike from demo account,1
duplicate export row,1
confidence missing as text,1
review policy changed mid-day,1


In [10]:
for col in ["team", "workflow", "source"]:
    normalized = df_raw[col].astype("string").str.strip().str.lower()

    inconsistent = (
        pd.DataFrame({"raw": df_raw[col], "normalized": normalized})
        .drop_duplicates()
        .groupby("normalized")["raw"]
        .agg(list)
        .reset_index()
    )

    inconsistent["raw_label_count"] = inconsistent["raw"].apply(len)

    print(f"\n### Potential label inconsistencies: {col}")
    display(inconsistent.query("raw_label_count > 1"))


### Potential label inconsistencies: team


,normalized,raw,raw_label_count
0,product,"[Product, product]",2



### Potential label inconsistencies: workflow


,normalized,raw,raw_label_count



### Potential label inconsistencies: source


,normalized,raw,raw_label_count


#Validate logical count relationships

In [11]:
df_check = df_raw.copy()
df_check["date"] = pd.to_datetime(df_check["date"], errors="coerce")

count_issues = df_check[
    (df_check["completed"] > df_check["sessions"]) |
    (df_check["accepted_output"] > df_check["completed"]) |
    (df_check["flagged_for_review"] > df_check["completed"]) |
    (df_check[["sessions", "completed", "accepted_output", "flagged_for_review"]] < 0).any(axis=1)
]

range_issues = df_check[
    (~df_check["median_confidence"].between(0, 1)) & df_check["median_confidence"].notna()
    |
    (~df_check["user_rating"].between(1, 5)) & df_check["user_rating"].notna()
    |
    (df_check["avg_minutes_saved"] < 0)
]

print(f"Invalid count relationships: {len(count_issues)}")
display(count_issues)

print(f"Invalid value ranges: {len(range_issues)}")
display(range_issues)

Invalid count relationships: 0


,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes


Invalid value ranges: 0


,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes


#Look for suspicious spikes

In [12]:
suspicious_rows = df_check[
    df_check["notes"].str.contains(
        "spike|duplicate|missing|policy|small sample|casing",
        case=False,
        na=False
    )
].sort_values("date")

print(f"Rows with explicit data-quality or comparability caveats: {len(suspicious_rows)}")
display(
    suspicious_rows[
        ["date", "team", "workflow", "source", "sessions",
         "completed", "accepted_output", "flagged_for_review",
         "median_confidence", "user_rating", "notes"]
    ]
)

Rows with explicit data-quality or comparability caveats: 8


,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,median_confidence,user_rating,notes
3,2026-08-01,Support,Reply draft,manual,11,8,5,1,0.68,NaN,missing rating
4,2026-08-01,Product,Feedback clustering,csv upload,12,9,6,2,0.59,3.6,small sample
5,2026-08-01,Product,Feedback clustering,manual,5,4,3,1,0.55,3.5,small sample
11,2026-08-02,product,Feedback clustering,manual,6,4,2,1,0.52,3.4,team casing differs
24,2026-08-05,Sales,Lead summary,email,140,126,119,2,0.95,4.9,traffic spike from demo account
25,2026-08-05,Sales,Lead summary,email,140,126,119,2,0.95,4.9,duplicate export row
30,2026-08-05,Product,Feedback clustering,manual,9,7,4,1,NaN,3.5,confidence missing as text
38,2026-08-07,Support,Reply draft,queue,30,17,8,12,0.91,2.1,review policy changed mid-day


## Data-quality findings and handling decisions

The raw export contains 41 daily workflow records across seven dates. The dataset
is small and observational, so this analysis is intended to prioritize
investigations rather than establish causal impact.

I found one missing user rating and one missing confidence value. I retain those
rows because the underlying usage counts remain valid, but the app will display
rating and confidence coverage alongside the metrics.

I normalize `Product` and `product` to a single team label. I identified one
likely duplicate export: two Aug. 5 Sales / Lead Summary records have identical
operational values but different notes. I exclude the row marked `duplicate
export row` from aggregated metrics while preserving the issue as a data-quality
warning.

I also exclude the demo-account traffic spike from normal health comparisons and
exclude the mid-day review-policy-change record from review-flag comparisons.
These decisions prevent known non-product or non-comparable events from
distorting the health-check recommendation.

#Create clean analysis dataset

In [13]:
df_clean = df_raw.copy()

df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean["team"] = df_clean["team"].str.strip().str.title()
df_clean["workflow"] = df_clean["workflow"].str.strip()
df_clean["source"] = df_clean["source"].str.strip()
df_clean["notes"] = df_clean["notes"].str.strip()

df_clean["is_duplicate_export"] = (
    df_clean["notes"]
    .str.contains("duplicate export", case=False, na=False)
)

df_clean["is_demo_traffic"] = (
    df_clean["notes"]
    .str.contains("demo account", case=False, na=False)
)

df_clean["review_policy_changed"] = (
    df_clean["notes"]
    .str.contains("review policy changed", case=False, na=False)
)

df_clean["is_small_sample"] = (
    df_clean["notes"]
    .str.contains("small sample", case=False, na=False)
)

df_metrics = df_clean[
    ~df_clean["is_duplicate_export"] &
    ~df_clean["is_demo_traffic"]
].copy()

print(f"Raw rows: {len(df_raw)}")
print(f"Rows usable for core workflow metrics: {len(df_metrics)}")
print(f"Excluded duplicate-export rows: {df_clean['is_duplicate_export'].sum()}")
print(f"Excluded demo-traffic rows: {df_clean['is_demo_traffic'].sum()}")

display(
    df_clean[
        df_clean[
            ["is_duplicate_export", "is_demo_traffic",
             "review_policy_changed", "is_small_sample"]
        ].any(axis=1)
    ][
        ["date", "team", "workflow", "source", "sessions",
         "completed", "accepted_output", "flagged_for_review",
         "notes", "is_duplicate_export", "is_demo_traffic",
         "review_policy_changed", "is_small_sample"]
    ]
)

Raw rows: 41
Rows usable for core workflow metrics: 39
Excluded duplicate-export rows: 1
Excluded demo-traffic rows: 1


,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,notes,is_duplicate_export,is_demo_traffic,review_policy_changed,is_small_sample
4,2026-08-01,Product,Feedback clustering,csv upload,12,9,6,2,small sample,False,False,False,True
5,2026-08-01,Product,Feedback clustering,manual,5,4,3,1,small sample,False,False,False,True
24,2026-08-05,Sales,Lead summary,email,140,126,119,2,traffic spike from demo account,False,True,False,False
25,2026-08-05,Sales,Lead summary,email,140,126,119,2,duplicate export row,True,False,False,False
38,2026-08-07,Support,Reply draft,queue,30,17,8,12,review policy changed mid-day,False,False,True,False


#Calculate weighted pre/post changes metric

#Coverage check

In [14]:
PROMPT_CHANGE_DATE = pd.Timestamp("2026-08-04")

df_metrics["period"] = np.where(
    df_metrics["date"] < PROMPT_CHANGE_DATE,
    "Pre-change",
    "Post-change"
)

display(
    df_metrics.groupby(["period", "workflow", "source"])
    .agg(
        days=("date", "nunique"),
        rows=("date", "size"),
        sessions=("sessions", "sum")
    )
    .reset_index()
    .sort_values(["workflow", "source", "period"])
)

,period,workflow,source,days,rows,sessions
0,Post-change,Feedback clustering,csv upload,4,4,103
6,Pre-change,Feedback clustering,csv upload,3,3,46
1,Post-change,Feedback clustering,manual,4,4,40
7,Pre-change,Feedback clustering,manual,3,3,18
2,Post-change,Lead summary,email,3,3,172
8,Pre-change,Lead summary,email,3,3,138
3,Post-change,Lead summary,manual,3,3,80
9,Pre-change,Lead summary,manual,3,3,60
4,Post-change,Reply draft,manual,3,3,48
10,Pre-change,Reply draft,manual,3,3,36


#Add metric calc function

In [15]:
def calculate_health_metrics(data):
    completed = data["completed"].sum()
    sessions = data["sessions"].sum()

    review_comparable = data[~data["review_policy_changed"]]
    review_completed = review_comparable["completed"].sum()

    rating_coverage = (
        data.loc[data["user_rating"].notna(), "sessions"].sum() / sessions
        if sessions > 0 else np.nan
    )

    confidence_coverage = (
        data.loc[data["median_confidence"].notna(), "sessions"].sum() / sessions
        if sessions > 0 else np.nan
    )

    return pd.Series({
        "days": data["date"].nunique(),
        "rows": len(data),
        "sessions": sessions,
        "completed": completed,
        "accepted_outputs": data["accepted_output"].sum(),
        "review_comparable_sessions": review_comparable["sessions"].sum(),
        "completion_rate": completed / sessions if sessions else np.nan,
        "acceptance_rate": data["accepted_output"].sum() / completed if completed else np.nan,
        "review_flag_rate": (
            review_comparable["flagged_for_review"].sum() / review_completed
            if review_completed else np.nan
        ),
        "avg_minutes_saved_per_completed": (
            np.average(
                data["avg_minutes_saved"],
                weights=data["completed"]
            ) if completed else np.nan
        ),
        "avg_user_rating": data["user_rating"].mean(),
        "rating_coverage_by_sessions": rating_coverage,
        "median_confidence_daily": data["median_confidence"].median(),
        "confidence_coverage_by_sessions": confidence_coverage,
        "policy_change_rows_excluded_from_review_rate": data["review_policy_changed"].sum(),
        "small_sample_rows": data["is_small_sample"].sum()
    })

#Create the pre/post comparison table


In [16]:
comparison = (
    df_metrics
    .groupby(["workflow", "source", "period"], group_keys=False)
    .apply(calculate_health_metrics, include_groups=False)
    .reset_index()
)

comparison_display = comparison.copy()

percentage_cols = [
    "completion_rate",
    "acceptance_rate",
    "review_flag_rate",
    "rating_coverage_by_sessions",
    "confidence_coverage_by_sessions"
]

for col in percentage_cols:
    comparison_display[col] = (comparison_display[col] * 100).round(1)

display(comparison_display.sort_values(["workflow", "source", "period"]))

,workflow,source,period,days,rows,sessions,completed,accepted_outputs,review_comparable_sessions,completion_rate,acceptance_rate,review_flag_rate,avg_minutes_saved_per_completed,avg_user_rating,rating_coverage_by_sessions,median_confidence_daily,confidence_coverage_by_sessions,policy_change_rows_excluded_from_review_rate,small_sample_rows
0,Feedback clustering,csv upload,Post-change,4.0,4.0,103.0,64.0,43.0,103.0,62.1,67.2,20.3,14.050000,3.850000,100.0,0.650,100.0,0.0,0.0
1,Feedback clustering,csv upload,Pre-change,3.0,3.0,46.0,32.0,22.0,46.0,69.6,68.8,18.8,13.453125,3.700000,100.0,0.600,100.0,0.0,1.0
2,Feedback clustering,manual,Post-change,4.0,4.0,40.0,29.0,18.0,40.0,72.5,62.1,13.8,11.151724,3.575000,100.0,0.570,77.5,0.0,0.0
3,Feedback clustering,manual,Pre-change,3.0,3.0,18.0,13.0,8.0,18.0,72.2,61.5,23.1,10.769231,3.466667,100.0,0.530,100.0,0.0,1.0
4,Lead summary,email,Post-change,3.0,3.0,172.0,139.0,112.0,172.0,80.8,80.6,8.6,8.698561,4.266667,100.0,0.800,100.0,0.0,0.0
5,Lead summary,email,Pre-change,3.0,3.0,138.0,113.0,92.0,138.0,81.9,81.4,8.8,8.363717,4.166667,100.0,0.760,100.0,0.0,0.0
6,Lead summary,manual,Post-change,3.0,3.0,80.0,57.0,42.0,80.0,71.2,73.7,10.5,6.663158,4.000000,100.0,0.670,100.0,0.0,0.0
7,Lead summary,manual,Pre-change,3.0,3.0,60.0,41.0,27.0,60.0,68.3,65.9,14.6,6.173171,3.833333,100.0,0.630,100.0,0.0,0.0
8,Reply draft,manual,Post-change,3.0,3.0,48.0,35.0,24.0,48.0,72.9,68.6,14.3,3.465714,3.866667,100.0,0.730,100.0,0.0,0.0
9,Reply draft,manual,Pre-change,3.0,3.0,36.0,27.0,17.0,36.0,75.0,63.0,11.1,3.214815,3.750000,69.4,0.690,100.0,0.0,0.0


In [17]:
metrics_for_comparison = [
    "days",
    "sessions",
    "completion_rate",
    "acceptance_rate",
    "review_flag_rate",
    "avg_minutes_saved_per_completed",
    "avg_user_rating",
    "rating_coverage_by_sessions",
    "median_confidence_daily"
]

before_after = (
    comparison
    .pivot(
        index=["workflow", "source"],
        columns="period",
        values=metrics_for_comparison
    )
)

before_after.columns = [
    f"{metric}_{period.lower().replace('-', '_')}"
    for metric, period in before_after.columns
]

before_after = before_after.reset_index()

for metric in [
    "completion_rate",
    "acceptance_rate",
    "review_flag_rate",
    "avg_minutes_saved_per_completed",
    "avg_user_rating",
    "median_confidence_daily"
]:
    pre_col = f"{metric}_pre_change"
    post_col = f"{metric}_post_change"

    if pre_col in before_after and post_col in before_after:
        before_after[f"{metric}_change"] = (
            before_after[post_col] - before_after[pre_col]
        )

display(before_after.round(3))

,workflow,source,days_post_change,days_pre_change,sessions_post_change,sessions_pre_change,completion_rate_post_change,completion_rate_pre_change,acceptance_rate_post_change,acceptance_rate_pre_change,...,rating_coverage_by_sessions_post_change,rating_coverage_by_sessions_pre_change,median_confidence_daily_post_change,median_confidence_daily_pre_change,completion_rate_change,acceptance_rate_change,review_flag_rate_change,avg_minutes_saved_per_completed_change,avg_user_rating_change,median_confidence_daily_change
0,Feedback clustering,csv upload,4.0,3.0,103.0,46.0,0.621,0.696,0.672,0.688,...,1.0,1.000,0.650,0.60,-0.074,-0.016,0.016,0.597,0.150,0.050
1,Feedback clustering,manual,4.0,3.0,40.0,18.0,0.725,0.722,0.621,0.615,...,1.0,1.000,0.570,0.53,0.003,0.005,-0.093,0.382,0.108,0.040
2,Lead summary,email,3.0,3.0,172.0,138.0,0.808,0.819,0.806,0.814,...,1.0,1.000,0.800,0.76,-0.011,-0.008,-0.002,0.335,0.100,0.040
3,Lead summary,manual,3.0,3.0,80.0,60.0,0.712,0.683,0.737,0.659,...,1.0,1.000,0.670,0.63,0.029,0.078,-0.041,0.490,0.167,0.040
4,Reply draft,manual,3.0,3.0,48.0,36.0,0.729,0.750,0.686,0.630,...,1.0,0.694,0.730,0.69,-0.021,0.056,0.032,0.251,0.117,0.040
5,Reply draft,queue,4.0,3.0,246.0,180.0,0.793,0.856,0.759,0.792,...,1.0,1.000,0.875,0.83,-0.063,-0.033,0.021,-0.351,-0.383,0.045


In [18]:
readable_comparison = before_after.copy()

for col in readable_comparison.columns:
    if any(metric in col for metric in [
        "completion_rate",
        "acceptance_rate",
        "review_flag_rate"
    ]):
        readable_comparison[col] = (
            readable_comparison[col] * 100
        ).round(1)

display(readable_comparison)

,workflow,source,days_post_change,days_pre_change,sessions_post_change,sessions_pre_change,completion_rate_post_change,completion_rate_pre_change,acceptance_rate_post_change,acceptance_rate_pre_change,...,rating_coverage_by_sessions_post_change,rating_coverage_by_sessions_pre_change,median_confidence_daily_post_change,median_confidence_daily_pre_change,completion_rate_change,acceptance_rate_change,review_flag_rate_change,avg_minutes_saved_per_completed_change,avg_user_rating_change,median_confidence_daily_change
0,Feedback clustering,csv upload,4.0,3.0,103.0,46.0,62.1,69.6,67.2,68.8,...,1.0,1.000000,0.650,0.60,-7.4,-1.6,1.6,0.596875,0.150000,0.050
1,Feedback clustering,manual,4.0,3.0,40.0,18.0,72.5,72.2,62.1,61.5,...,1.0,1.000000,0.570,0.53,0.3,0.5,-9.3,0.382493,0.108333,0.040
2,Lead summary,email,3.0,3.0,172.0,138.0,80.8,81.9,80.6,81.4,...,1.0,1.000000,0.800,0.76,-1.1,-0.8,-0.2,0.334844,0.100000,0.040
3,Lead summary,manual,3.0,3.0,80.0,60.0,71.2,68.3,73.7,65.9,...,1.0,1.000000,0.670,0.63,2.9,7.8,-4.1,0.489987,0.166667,0.040
4,Reply draft,manual,3.0,3.0,48.0,36.0,72.9,75.0,68.6,63.0,...,1.0,0.694444,0.730,0.69,-2.1,5.6,3.2,0.250899,0.116667,0.040
5,Reply draft,queue,4.0,3.0,246.0,180.0,79.3,85.6,75.9,79.2,...,1.0,1.000000,0.875,0.83,-6.3,-3.3,2.1,-0.350556,-0.383333,0.045


In [19]:
before_after.to_csv('before_after.csv', index=False)

#Recommendation logic


In [20]:
MIN_SESSIONS_PER_PERIOD = 30

def assign_status(row):
    min_sessions = min(
        row["sessions_pre_change"],
        row["sessions_post_change"]
    )

    if min_sessions < MIN_SESSIONS_PER_PERIOD:
        return "Inconclusive"

    if (
        row["completion_rate_change"] <= -0.05 or
        row["acceptance_rate_change"] <= -0.03 or
        row["review_flag_rate_change"] >= 0.02
    ):
        return "Investigate"

    if (
        row["completion_rate_change"] >= -0.02 and
        row["acceptance_rate_change"] >= 0 and
        row["review_flag_rate_change"] <= 0.01
    ):
        return "Promising"

    return "Inconclusive"

In [21]:
def build_reasons(row):
    reasons = []

    min_sessions = min(
        row["sessions_pre_change"],
        row["sessions_post_change"]
    )

    if min_sessions < MIN_SESSIONS_PER_PERIOD:
        reasons.append(
            f"Low comparison volume: only {min_sessions:.0f} sessions "
            "in one of the two periods."
        )

    if row["completion_rate_change"] <= -0.05:
        reasons.append(
            f"Completion rate decreased by "
            f"{abs(row['completion_rate_change']) * 100:.1f} percentage points."
        )

    if row["acceptance_rate_change"] <= -0.03:
        reasons.append(
            f"Acceptance rate decreased by "
            f"{abs(row['acceptance_rate_change']) * 100:.1f} percentage points."
        )

    if row["acceptance_rate_change"] >= 0.03:
        reasons.append(
            f"Acceptance rate increased by "
            f"{row['acceptance_rate_change'] * 100:.1f} percentage points."
        )

    if row["review_flag_rate_change"] >= 0.02:
        reasons.append(
            f"Review-flag rate increased by "
            f"{row['review_flag_rate_change'] * 100:.1f} percentage points."
        )

    if row["review_flag_rate_change"] <= -0.02:
        reasons.append(
            f"Review-flag rate decreased by "
            f"{abs(row['review_flag_rate_change']) * 100:.1f} percentage points."
        )

    return reasons

In [22]:
before_after["status"] = before_after.apply(assign_status, axis=1)
before_after["reasons"] = before_after.apply(build_reasons, axis=1).apply(
    lambda items: " ".join(items) if items else "No material metric movement detected."
)

decision_table = before_after[
    [
        "workflow",
        "source",
        "status",
        "sessions_pre_change",
        "sessions_post_change",
        "completion_rate_change",
        "acceptance_rate_change",
        "review_flag_rate_change",
        "avg_user_rating_change",
        "reasons"
    ]
].copy()

for col in [
    "completion_rate_change",
    "acceptance_rate_change",
    "review_flag_rate_change"
]:
    decision_table[col] = (decision_table[col] * 100).round(1)

decision_table = decision_table.sort_values(
    by="status",
    key=lambda col: col.map({
        "Investigate": 0,
        "Inconclusive": 1,
        "Promising": 2
    })
)

display(decision_table)

,workflow,source,status,sessions_pre_change,sessions_post_change,completion_rate_change,acceptance_rate_change,review_flag_rate_change,avg_user_rating_change,reasons
0,Feedback clustering,csv upload,Investigate,46.0,103.0,-7.4,-1.6,1.6,0.150000,Completion rate decreased by 7.4 percentage po...
4,Reply draft,manual,Investigate,36.0,48.0,-2.1,5.6,3.2,0.116667,Acceptance rate increased by 5.6 percentage po...
5,Reply draft,queue,Investigate,180.0,246.0,-6.3,-3.3,2.1,-0.383333,Completion rate decreased by 6.3 percentage po...
1,Feedback clustering,manual,Inconclusive,18.0,40.0,0.3,0.5,-9.3,0.108333,Low comparison volume: only 18 sessions in one...
2,Lead summary,email,Inconclusive,138.0,172.0,-1.1,-0.8,-0.2,0.100000,No material metric movement detected.
3,Lead summary,manual,Promising,60.0,80.0,2.9,7.8,-4.1,0.166667,Acceptance rate increased by 7.8 percentage po...


In [23]:
policy_affected = (
    df_clean[df_clean["review_policy_changed"]]
    [["workflow", "source"]]
    .drop_duplicates()
)

decision_table = decision_table.merge(
    policy_affected.assign(review_policy_caveat=True),
    on=["workflow", "source"],
    how="left"
)

decision_table["review_policy_caveat"] = (
    decision_table["review_policy_caveat"].fillna(False)
)

decision_table.loc[
    decision_table["review_policy_caveat"],
    "reasons"
] += (
    " Review-flag interpretation remains limited because a review-policy "
    "change occurred during the post-change period."
)

display(decision_table)

/tmp/ipykernel_2942/3996899897.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  decision_table["review_policy_caveat"].fillna(False)


,workflow,source,status,sessions_pre_change,sessions_post_change,completion_rate_change,acceptance_rate_change,review_flag_rate_change,avg_user_rating_change,reasons,review_policy_caveat
0,Feedback clustering,csv upload,Investigate,46.0,103.0,-7.4,-1.6,1.6,0.150000,Completion rate decreased by 7.4 percentage po...,False
1,Reply draft,manual,Investigate,36.0,48.0,-2.1,5.6,3.2,0.116667,Acceptance rate increased by 5.6 percentage po...,False
2,Reply draft,queue,Investigate,180.0,246.0,-6.3,-3.3,2.1,-0.383333,Completion rate decreased by 6.3 percentage po...,True
3,Feedback clustering,manual,Inconclusive,18.0,40.0,0.3,0.5,-9.3,0.108333,Low comparison volume: only 18 sessions in one...,False
4,Lead summary,email,Inconclusive,138.0,172.0,-1.1,-0.8,-0.2,0.100000,No material metric movement detected.,False
5,Lead summary,manual,Promising,60.0,80.0,2.9,7.8,-4.1,0.166667,Acceptance rate increased by 7.8 percentage po...,False


## Pre/post health-check interpretation

The comparison is observational and should not be interpreted as causal evidence
that the prompt change produced the metric movements. The available data covers
only three pre-change days and three to four post-change days, with no control
group.

Lead Summary via manual use is the most promising directional signal: acceptance
increased by 7.8 percentage points, completion increased by 2.9 points, and
review flags decreased by 4.1 points.

Reply Draft via queue is the highest-priority investigation: completion declined
by 6.3 percentage points, acceptance declined by 3.3 points, and user rating
declined by 0.38. Review flags also increased, although the Aug. 7 review-policy
change makes that particular comparison less reliable.

Feedback Clustering via manual use is inconclusive because the pre-change period
contains only 18 sessions.

In [24]:
decision_table.to_csv('decision_table.csv', index=False)

## Final scope

This analysis supports a small Streamlit health-check tool for SignalDesk product
teammates. The tool compares pre- and post-prompt-change workflow metrics by
workflow and source, highlights data-quality caveats, and labels each comparison
as Promising, Investigate, or Inconclusive.

It is a decision-support tool, not a causal evaluation. The recommended next
action is to investigate Reply Draft via queue before broader rollout.